# want to code along with me? 
- download https://www.kaggle.com/snap/amazon-fine-food-reviews
- `pip install polars pyarrow altair vegafusion vl-convert-python graphviz`

# why polars?
<img src="assets/syntax.png" alt="why polars" style="width: 100%; height: auto;">


more on why polars is the pandas killer:<br>
<a href="https://www.youtube.com/watch?v=sepiszMSvBs">
<img src="assets/polars_is pandas_killer.png" alt="Thy polars2" style="width: 50%; height: auto;">
</a>

# Gradual exposure therapy

<img src="assets/therapy.webp" alt="Therapy Image" style="width: 25%; height: auto;">


# Same API as Pandas

<img src="assets/mirror.png" alt="Therapy Image" style="width: 25%; height: auto;">

# read
.read_csv()<br>
.read_excel()<br>
.read_json()<br>
.read_parquet()<br>

# inspect
.head()<br>
.tail()<br>
.describe()<br>
.sample()<br>
.shape<br>
.columns<br>
.dtypes<br>
.value_counts()<br>

# .str
.replace()<br>
.contains<br>
.split()<br>

# math
.sum()<br>
.min()<br>
.max()<br>
.round()<br>
.sqrt()<br>
.corr()<br>

# other
.DataFrame()<br>
.concat()<br>
.shift()<br>
.pipe()<br>
.cut()<br>
.to_list()<br>
.pivot()<br>
.explode()<br>
.first()<br>
.melt()<br>
.rename()<br>
.rolling()<br>
.plot()<br>


In [ ]:
import polars as pl

In [ ]:
# create a dataframe from a dictionary
df = pl.DataFrame({'col_a': [1,2,3],
                   'col_b': [4,5,6]})
df

In [ ]:
# download Reviews.csv from https://www.kaggle.com/snap/amazon-fine-food-reviews
df = pl.read_csv("Reviews.csv")
df.head()

In [ ]:
df.dtypes

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
# column selection (more to come)
df['ProfileName']

In [ ]:
df[['ProfileName', 'UserId']]

In [ ]:
df['ProfileName'][0]

In [ ]:
df = df.rename({'Id': 'id'})
df.sample(2)

In [ ]:
df.describe()

In [ ]:
df['ProfileName'].value_counts()

# plotting
- Built-in plotting with Altair
- You may slelct plotting with hvplot:
```
import hvplot.polars
df.hvplot.scatter(
    x="sepal_width",
    y="sepal_length",
    by="species",
    width=650,
    title="Irises",
    xlabel='Sepal Width',
    ylabel='Sepal Length',
)
```
- or just pass the dataframe/columns to matplotlib, seaborn or plotly<br><br>
for plotting we need to add a few packages:
- `poetry add altair`
- `poetry add vegafusion` (for plotting large datasets)
- `poetry add vl-convert-python`

In [ ]:
import altair as alt
alt.data_transformers.enable("vegafusion")

df['Score'].plot.hist()

In [ ]:
df.plot.point(
        x="Score",
        y="HelpfulnessNumerator",
    ).properties(
        width=500,
        title="Score-HelpfulnessNumerator scatterplot"
        )

# similar API to pandas

<img src="assets/similar.png" alt="Therapy Image" style="width: 25%; height: auto;">

| Pandas          | Polars           |
|-----------------|------------------|
| `.groupby()`    | `.group_by()`    |
| `.isin()`       | `.is_in()`       |
| `.nunique()`    | `.n_unique()`    |
| `.drop(axis=1)` | `.drop()`        |
| `.to_csv()`     | `.write_csv()`   |
| `.isna()`       | `.is_null()`     |
| `.notna()`      | `.is_not_null()` |
| `.dropna()`     | `.drop_nulls()`  |
| `.sort_values()`| `.sort()`        |


Some polars conventions:
- always put an undersocre between words od a method.
- .write_xyz is used to create files (csv, parquet, ...)
- .to_abc is used for conversions (to_list, to_pandas, to_numpy, ...)
- polars (generally) uses `Null` to represent a missing value (nan still can be used for a different purpose)
- polars does not have an index column (can be added using `df.with_row_index()`)

In [ ]:
# add Null values "randomly". Ignore for now
df = (
    df
    .with_columns(
        Score=pl.when(pl.col("Id") % 23 ==0).then(pl.lit(None)).otherwise(pl.col.Score)
    )
)

# how many missing values in 'Score' column?
df['Score'].is_null().sum()         # pandas: .isna().sum()

In [ ]:
df = (
    df
    .drop('index')                       # pandas: .drop('index', axis=1)
    .drop_nulls('Score')                 # pandas: .dropna(subset='Score')
    .sort(by='Score', descending=True)   # pandas: .sort_values(by='Score', ascending=False)
)
df

# different API 1

<img src="assets/different.png" alt="Therapy Image" style="width: 25%; height: auto;">

| Pandas          | Polars           |
|-----------------|------------------|
| `.T`            |`.transpose()`    |
| `.merge()`      |`.join()`         |
|`.drop_duplicates()`|`.unique()`    |
|`.values()`      |`.item()`         |

In [ ]:
df.head()

In [ ]:
df['HelpfulnessNumerator'].value_counts()

In [ ]:
df['HelpfulnessNumerator'].value_counts().transpose(include_header=True)

In [ ]:
df.head()

In [ ]:
df.unique('UserId') # pandas: .drop_duplicates(subset='UserId')

In [ ]:
df.item(1,'ProfileName')

# expressions and context
polars uses "expresions" that can be optimized. The explressions will be in a context

Selection-------->df.select(...)<br>
manipulation----->df.with_columns(...)<br>
Filtering--------->df.filter()<br>
Group by/Agg---->df.group_by(...).agg(...)<br>

basic expresion: `pl.col` refers to a column (we can use it in any context)

In [ ]:
pl.col('Text')

In [ ]:
type(pl.col('Text'))

# What's the point of expressions? Query optimization
<img src="assets/query_optimization.png" alt="Therapy Image" style="width: 50%; height: auto;"><br>
we will talk about it soon, but first, let's get used to it:

In [ ]:
df.select(pl.col('Text'))

In [ ]:
(
    df
    .filter(pl.col('Score') > 3)                        # pandas: df[df['Score'] > 3]
    .select(pl.col('ProfileName'), pl.col('Score'))
    .head()
)

In [ ]:
(
    df
    .filter(pl.col('Summary').str.contains('good'))
    # alternative:
    .select(pl.col.ProfileName, pl.col.Score)
    .head()
)

In [ ]:
# possible, but not recommended
from polars import col # as c
df.filter(
    col.Summary.str.contains('good')
)
# for col in df.columns...

# different API 2
| Pandas          | Polars           |
|-----------------|------------------|
|`.assign()`      |`.with_columns()` |
|`.as_type()`     |`.cast()`         |
|`.apply()`       |`.map_elements()` |

In [ ]:
(
    df
    .with_columns(                                                      # pandas: .assign(..)
        pl.col('ProfileName').str.to_lowercase().alias('profile_name'), # pandas: .str.lower() 
        new_score=pl.col('Score') * 2,
        Score=pl.col('Score').cast(pl.Int32)                            # pandas: .astype(int)
        )
    .select('ProfileName', 'profile_name', 'Score', 'new_score')
    .head()
)


In [ ]:
# when using .map_elements (pandas: .apply) we need to specify the return type:
(
    df
    .with_columns(
        reverse_name=pl.col('ProfileName')
                .map_elements(lambda s: s[::-1], return_dtype=pl.String)) # pandas: .apply(lambda s: s[::-1])
    .select('ProfileName', 'reverse_name')    
    .head()
)

<img src="assets/polars_types.png" alt="Polars data types" style="width: 50%; height: auto;">

[more on polars data types](https://pola.rs/posts/understanding-polars-data-types/)

In [ ]:
# automatic type inference and conversion
(
    df
    .select(
        pl.all().shrink_dtype() # pandas: .convert_dtypes()
        )
)

# cool stuff
`.glimpse()`<br>
`.schema`<br>
`.shrink_dtypes()`<br>
`.list()`<br>
`pl.when(...).then(...).otherwise(..)`<br>
`.over()`<br>
`.unnest()`



In [ ]:
df.glimpse()

In [ ]:
df.schema

In [ ]:
# group by ProductId, put all texts in a list
grouped = df.group_by('ProductId').agg(pl.col('Text'))
grouped.head(2)

In [ ]:
grouped.with_columns(pl.col("Text").list.len())

In [ ]:
(
    grouped
    .filter(pl.col("Text").list.len()>40)
    .with_columns(pl.col("Text").list.get(0))    
)

In [ ]:
(
    grouped
    .with_columns(
        commenter_type=
            pl.when(pl.col("Text").list.len()>=40).then(pl.lit("full_time_commenter"))
            .when(
                (pl.col("Text").list.len()<40) & (pl.col("Text").list.len()>1)
                ).then(pl.lit("medium"))
            .when(pl.col("Text").list.len()==1).then(pl.lit("one_timer"))
            .otherwise(pl.lit("inspect"))
        )
)

# similatr to pandas' use of np.where() or np.select()

In [ ]:
# over is like groupby+merge in pandas (Compute expressions over the given groups)
df.with_columns(avg_user_score=pl.col('Score').mean().over('UserId'))

In [ ]:
nested_df = pl.DataFrame({
    "person": [{"data": {"name": "igor", "age": 28}},{"data": {"name": "james", "age": 30}}, {"data": {"name": "john", "age": 40}}, {"data": {"name": "jane", "age": 50}},],
    "love": [{"data": {"food": "steak", "sport": "powerlifting"}}, {"data": {"food": "pizza", "sport": "running"}}, {"data": {"food": "sushi", "sport": "swimming"}}, {"data": {"food": "pasta", "sport": "cycling"}},],
})
nested_df

In [ ]:
nested_df.unnest("person")

In [ ]:
nested_df.unnest("person").unnest("data")

# Eager mode vs Lazy mode
<img src="assets/lazy_bill.png" alt="Polars data types" style="width: 50%; height: auto;">

- `pl.scan_scv()....collect()`
- `pl.scan_parquet()....collect()`
- `df.lazy().....collect()`

In [ ]:
df = pl.scan_csv('Reviews.csv')
df

In [ ]:
type(df)

In [ ]:
df = (
    pl.scan_csv('Reviews.csv')
    .filter(pl.col("Score") > 3)
    .group_by('ProductId').agg(pl.col('Text'))
    .with_columns(
        commenter_type=pl.when(pl.col("Text").list.len()>=40).then(pl.lit("full_time_commenter"))
            .when(
                (pl.col("Text").list.len()<40) & (pl.col("Text").list.len()>1)
                ).then(pl.lit("medium"))
            .when(pl.col("Text").list.len()==1).then(pl.lit("one_timer"))
            .otherwise(pl.lit("inspect"))
        )
    .filter(pl.col("commenter_type").is_in(["full_time_commenter", "medium"]))
    .sort('ProductId')
)
df

In [ ]:
print(df.explain(optimized=True))

In [ ]:
print(df.explain())

In [ ]:
df.show_graph()

In [ ]:
df.show_graph(optimized=True)

In [ ]:
df, profile = df.profile()
profile

In [ ]:
profile.with_columns(diff = pl.col('end') - pl.col('start'))

In [ ]:
df.collect()

# Collection engine: Streaming/GPU
- By default, Polars dataframes are limited to ~4.3 billion. Increase this limit to (~18 quintillion) by enabling the big index extension:`pip install polars-u64-idx`
- If our computer doesn't have enogh RAM for dataframe stroage, we can read and manipulate in chunks using `.collect(streaming=True)`
- If the output of a query is still too large to fit in memory we can write the output directly to a file on disk using sink methods (`.sink_csv()`, `.sink_parquet()` ect)
- `df = pl.scan_csv('input.csv')......sink_csv('output.csv')`
- to allow collection using GPU: `pip install polars[gpu]`, `.collect(engine='gpu')`
- [more on streaming](https://www.rhosignal.com/posts/streaming-in-polars/)
- [more on gpu](https://pola.rs/posts/gpu-engine-release/)


# when do i still use pandas? 
<img src="assets/to_pandas.png" alt="Polars data types" style="width: 25%; height: auto;">

- polars won't save csv's with nested data (it can save parquets)-->`df.to_pandas().to_csv(...)`
- polars doesn't have an index and multi index, so some manipulation require more effort (`.corr()`, `.crosstab()`)

In [ ]:
df = pl.read_csv('Reviews.csv')
df.select("HelpfulnessNumerator",
          "HelpfulnessDenominator",
          "Score").corr()#.with_columns(ind=pl.Series(["HelpfulnessNumerator", "HelpfulnessDenominator", "Score"]))


# who to follow on polars?
- Ritchie Vink (the author)
- Liam Brannigan (has a Udemy course on polars)
- Matt Harrison (wrote "effective polars" and "effective pandas")
- "Israel Polars Fans" on linkedin